# Lekcja 13: SQL — Zadania

Wszystkie tabele tworzone **raz na górze** w jednym połączeniu `conn`.
Zadania 8, 16, 20 (benchmarki) używają osobnych połączeń z dużą liczbą wierszy.

## Setup — tabele

| Tabela | Wierszy | Zadania |
|---|---|---|
| `products` | 10 | 1, 2, 3, 7, 12 |
| `customers` | 5 | 4, 5, 6, 9 |
| `orders` | 7 | 4, 5, 6, 9 |
| `order_items` | 7 | 9 |
| `employees` | 8 | 18 |
| `sales` | 1 000 | 10, 13, 15, 19 |
| `customers_big` | 100 | 14, 17 |
| `orders_big` | 500 | 14, 17 |
| `passengers` | 891 | 11 |

In [1]:
import sqlite3, pandas as pd, numpy as np, matplotlib.pyplot as plt, time
%matplotlib inline
np.random.seed(42)
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

conn = sqlite3.connect(':memory:')

# products
pd.DataFrame({
    'product_id': range(1, 11),
    'name':     ['Laptop HP','Mysz Logitech','Klawiatura mech.','Monitor 27"',
                 'Sluchawki Sony','Kamera IP','Router Wi-Fi 6','SSD 1TB',
                 'Hub USB-C','Webcam HD'],
    'category': ['Elektronika','Akcesoria','Akcesoria','Elektronika','Audio',
                 'Sprzet','Sprzet','Komponenty','Akcesoria','Sprzet'],
    'price':    [3500, 80, 250, 1200, 350, 600, 280, 320, 90, 150],
    'stock':    [12, 150, 80, 25, 60, 20, 45, 90, 200, 70],
}).to_sql('products', conn, index=False)

# customers + orders
pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Jan Kowalski','Anna Nowak','Piotr Lis','Maria Zajac','Tomasz Woz'],
    'city': ['Warszawa','Krakow','Gdansk','Wroclaw','Poznan'],
}).to_sql('customers', conn, index=False)

pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105, 106, 107],
    'customer_id': [1,   2,   1,   3,   1,   2,   3],
    'order_date':  ['2024-01-10','2024-01-11','2024-02-05',
                   '2024-02-10','2024-03-01','2024-03-15','2024-03-20'],
    'amount':      [250.0, 120.5, 340.0, 80.0, 190.0, 450.0, 220.0],
}).to_sql('orders', conn, index=False)

# order_items
pd.DataFrame({
    'item_id':  [1, 2, 3, 4, 5, 6, 7],
    'order_id': [101,101,102,102,103,104,107],
    'product':  ['Laptop','Mysz','Klawiatura','Monitor','Sluchawki','Kamera','SSD'],
    'quantity': [1, 2, 1, 1, 2, 1, 1],
    'price':    [3500,80,250,1200,350,600,320],
}).to_sql('order_items', conn, index=False)

# employees
pd.DataFrame({
    'employee_id': [1,2,3,4,5,6,7,8],
    'name':  ['Anna Dyrektor','Piotr CFO','Marek CTO','Ewa Dev Lead',
              'Jan Analityk','Sara Developer','Adam Ksiegowy','Lena QA'],
    'title': ['CEO','CFO','CTO','Dev Lead','Analityk','Developer','Ksiegowy','QA'],
    'manager_id': [None,1,1,3,2,4,2,4],
}).to_sql('employees', conn, index=False)

# sales (1000 wierszy z datami 2024)
dates = pd.date_range('2024-01-01','2024-12-31').astype(str)
pd.DataFrame({
    'sale_id':   range(1, 1001),
    'sale_date': np.random.choice(dates, 1000),
    'product':   np.random.choice(['Laptop','Mysz','Monitor','Klawiatura','Sluchawki'],1000),
    'category':  np.random.choice(['Elektronika','Akcesoria','Audio'],1000),
    'revenue':   np.random.randint(50, 3500, 1000),
}).to_sql('sales', conn, index=False)

# customers_big (100) + orders_big (500)
pd.DataFrame({
    'customer_id':       range(1, 101),
    'name':              [f'Klient_{i}' for i in range(1,101)],
    'registration_date': pd.date_range('2024-01-01',periods=100,freq='3D').astype(str),
    'country':           np.random.choice(['Polska','Niemcy','Francja','UK','Czechy'],100),
}).to_sql('customers_big', conn, index=False)

pd.DataFrame({
    'order_id':    range(1, 501),
    'customer_id': np.random.randint(1, 101, 500),
    'order_date':  pd.date_range('2024-01-01',periods=500,freq='12h').astype(str),
    'amount':      np.random.uniform(50, 2000, 500).round(2),
}).to_sql('orders_big', conn, index=False)

# passengers (Titanic)
pd.read_csv(TITANIC_URL).to_sql('passengers', conn, index=False)

print('Tabele gotowe:')
for t in pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",conn)['name']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}',conn).iloc[0,0]
    print(f'  {t:<20} {n:>5} wierszy')

Tabele gotowe:
  products                10 wierszy
  customers                5 wierszy
  orders                   7 wierszy
  order_items              7 wierszy
  employees                8 wierszy
  sales                 1000 wierszy
  customers_big          100 wierszy
  orders_big             500 wierszy
  passengers             891 wierszy


---
## ✏️ Zadania podstawowe (1–8)

### ✏️ Zadanie 1 – Podstawowy SELECT

Utwórz bazę danych SQLite z tabelą `products` zawierającą kolumny: `product_id`, `name`, `category`, `price`, `stock`. Wstaw 10 przykładowych produktów.

**Wymagania:**
- Napisz zapytanie SELECT wybierające wszystkie produkty
- Napisz zapytanie wybierające tylko `name` i `price`
- Wyświetl wyniki za pomocą `pd.read_sql_query()`

*(proste)*

In [3]:
all_products = pd.read_sql_query("SELECT * FROM products", conn)
n_p_products = pd.read_sql_query("SELECT name, price FROM products", conn)

all_products, n_p_products

(   product_id              name     category  price  stock
 0           1         Laptop HP  Elektronika   3500     12
 1           2     Mysz Logitech    Akcesoria     80    150
 2           3  Klawiatura mech.    Akcesoria    250     80
 3           4       Monitor 27"  Elektronika   1200     25
 4           5    Sluchawki Sony        Audio    350     60
 5           6         Kamera IP       Sprzet    600     20
 6           7    Router Wi-Fi 6       Sprzet    280     45
 7           8           SSD 1TB   Komponenty    320     90
 8           9         Hub USB-C    Akcesoria     90    200
 9          10         Webcam HD       Sprzet    150     70,
                name  price
 0         Laptop HP   3500
 1     Mysz Logitech     80
 2  Klawiatura mech.    250
 3       Monitor 27"   1200
 4    Sluchawki Sony    350
 5         Kamera IP    600
 6    Router Wi-Fi 6    280
 7           SSD 1TB    320
 8         Hub USB-C     90
 9         Webcam HD    150)

### ✏️ Zadanie 2 – WHERE z warunkami

Używając tabeli `products` z zadania 1:

**Wymagania:**
- Znajdź produkty droższe niż 100 PLN
- Znajdź produkty z kategorii "Elektronika" LUB z ceną < 50 PLN
- Znajdź produkty z zapasem (`stock`) między 10 a 50

*(proste)*

In [7]:
over_100 = pd.read_sql_query("SELECT * FROM products WHERE price > 100", conn)
electronics_or_cheap = pd.read_sql_query("SELECT * FROM products WHERE category = 'Elektronika' OR price < 50", conn)
stocked = pd.read_sql_query("SELECT * FROM products WHERE stock BETWEEN 10 AND 50", conn)

over_100, electronics_or_cheap, stocked

(   product_id              name     category  price  stock
 0           1         Laptop HP  Elektronika   3500     12
 1           3  Klawiatura mech.    Akcesoria    250     80
 2           4       Monitor 27"  Elektronika   1200     25
 3           5    Sluchawki Sony        Audio    350     60
 4           6         Kamera IP       Sprzet    600     20
 5           7    Router Wi-Fi 6       Sprzet    280     45
 6           8           SSD 1TB   Komponenty    320     90
 7          10         Webcam HD       Sprzet    150     70,
    product_id         name     category  price  stock
 0           1    Laptop HP  Elektronika   3500     12
 1           4  Monitor 27"  Elektronika   1200     25,
    product_id            name     category  price  stock
 0           1       Laptop HP  Elektronika   3500     12
 1           4     Monitor 27"  Elektronika   1200     25
 2           6       Kamera IP       Sprzet    600     20
 3           7  Router Wi-Fi 6       Sprzet    280     45)

### ✏️ Zadanie 3 – ORDER BY i LIMIT

**Wymagania:**
- Wyświetl 5 najdroższych produktów
- Wyświetl 5 produktów z najmniejszym zapasem
- Wyświetl produkty posortowane alfabetycznie po nazwie

*(proste)*

In [9]:
most_expensive = pd.read_sql_query("SELECT * FROM products ORDER BY price DESC LIMIT 5", conn)
least_stocked = pd.read_sql_query("SELECT * FROM products ORDER BY stock ASC LIMIT 5", conn)
alphabetical = pd.read_sql_query("SELECT * FROM products ORDER BY name ASC", conn)

most_expensive, least_stocked, alphabetical

(   product_id            name     category  price  stock
 0           1       Laptop HP  Elektronika   3500     12
 1           4     Monitor 27"  Elektronika   1200     25
 2           6       Kamera IP       Sprzet    600     20
 3           5  Sluchawki Sony        Audio    350     60
 4           8         SSD 1TB   Komponenty    320     90,
    product_id              name    category  price  stock
 0           9         Hub USB-C   Akcesoria     90    200
 1           2     Mysz Logitech   Akcesoria     80    150
 2           8           SSD 1TB  Komponenty    320     90
 3           3  Klawiatura mech.   Akcesoria    250     80
 4          10         Webcam HD      Sprzet    150     70,
    product_id              name     category  price  stock
 0           9         Hub USB-C    Akcesoria     90    200
 1           6         Kamera IP       Sprzet    600     20
 2           3  Klawiatura mech.    Akcesoria    250     80
 3           1         Laptop HP  Elektronika   3500    

### ✏️ Zadanie 4 – Podstawowy JOIN

Utwórz dwie tabele: `customers` (customer_id, name, city) i `orders` (order_id, customer_id, amount).

**Wymagania:**
- Wykonaj INNER JOIN łączący klientów z zamówieniami
- Wyświetl nazwę klienta, miasto i kwotę zamówienia
- Posortuj wyniki według kwoty malejąco

*(proste)*

In [16]:
customers = pd.read_sql_query("SELECT * FROM customers", conn)
orders = pd.read_sql_query("SELECT order_id, customer_id, amount FROM orders", conn)

customer_orders = pd.read_sql_query("""
    SELECT c.name, c.city, o.amount
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY o.amount DESC
""", conn)

customer_orders

,name,city,amount
0,Anna Nowak,Krakow,450.0
1,Jan Kowalski,Warszawa,340.0
2,Jan Kowalski,Warszawa,250.0
3,Piotr Lis,Gdansk,220.0
4,Jan Kowalski,Warszawa,190.0
5,Anna Nowak,Krakow,120.5
6,Piotr Lis,Gdansk,80.0


### ✏️ Zadanie 5 – LEFT JOIN

Używając tabel z zadania 4:

**Wymagania:**
- Wykonaj LEFT JOIN pokazujący WSZYSTKICH klientów (nawet bez zamówień)
- Użyj COALESCE aby zamienić NULL na 0 w kwocie zamówienia
- Znajdź klientów którzy NIE złożyli zamówienia (`WHERE order_id IS NULL`)

*(proste)*

### ✏️ Zadanie 6 – Podstawowy GROUP BY

Używając tabeli `orders`:

**Wymagania:**
- Policz liczbę zamówień dla każdego klienta
- Oblicz łączną kwotę zamówień dla każdego klienta
- Oblicz średnią wartość zamówienia dla każdego klienta

*(proste)*

### ✏️ Zadanie 7 – COUNT i SUM

**Wymagania:**
- Policz łączną liczbę produktów w tabeli `products`
- Oblicz łączną wartość zapasów (`stock * price`) dla wszystkich produktów
- Oblicz średnią cenę produktów w każdej kategorii

*(proste)*

### ✏️ Zadanie 8 – Tworzenie indeksu

**Wymagania:**
- Utwórz tabelę z 10 000 rekordami (użyj pętli lub NumPy)
- Zmierz czas zapytania SELECT z filtrem WHERE **BEZ** indeksu
- Utwórz indeks na kolumnie filtrowanej
- Zmierz czas zapytania **Z** indeksem
- Porównaj wyniki

*(proste)*

> Osobne połączenie `conn_bench` — nie zaśmieca głównej bazy.

---
## ✏️ Zadania średnie (9–12)

### ✏️ Zadanie 9 – JOIN 3 tabel

Utwórz bazę sklepu: `customers`, `orders`, `order_items` (zawiera szczegóły produktów w zamówieniu).

**Wymagania:**
- Połącz 3 tabele w jednym zapytaniu
- Wyświetl nazwę klienta, datę zamówienia, nazwę produktu i cenę
- Oblicz łączną wartość każdego zamówienia
- Znajdź 10 największych zamówień

*(średnie)*

### ✏️ Zadanie 10 – HAVING i grupowanie

**Wymagania:**
- Używając danych sprzedażowych, pogrupuj według kategorii
- Użyj HAVING aby znaleźć tylko kategorie z przychodem > 1000 PLN
- Oblicz liczbę transakcji, łączny przychód i średnią dla każdej kategorii
- Posortuj według łącznego przychodu

*(średnie)*

### ✏️ Zadanie 11 – Złożone filtrowanie

**Dataset:** Titanic

**Wymagania:**
- Znajdź pasażerów 1 klasy, kobiety, które przeżyły
- Oblicz średni wiek i średnią cenę biletu dla tej grupy
- Porównaj z mężczyznami z 3 klasy którzy nie przeżyli
- Stwórz wykres porównawczy (barplot)

*(średnie)*

### ✏️ Zadanie 12 – Subquery (podzapytanie)

**Wymagania:**
- Znajdź produkty droższe niż średnia cena wszystkich produktów
- Użyj subquery: `WHERE price > (SELECT AVG(price) FROM products)`
- Wyświetl nazwę, cenę i różnicę od średniej
- Oblicz ile % to stanowi powyżej średniej

*(średnie)*

In [45]:
above_avg = pd.read_sql_query("""
    SELECT
    name,
    price,
    price - (SELECT AVG(price) FROM products) AS diff_from_avg,
    (
        SELECT COUNT(*) * 100.0 / (SELECT COUNT(*) FROM products)
        FROM products
        WHERE price > (SELECT AVG(price) FROM products)
    ) AS percentage_above_avg
FROM products
WHERE price > (SELECT AVG(price) FROM products);
""", conn)

above_avg

DatabaseError: Execution failed on sql '
SELECT
    name,
    price,
    price - (SELECT AVG(price) FROM products) AS diff_from_avg,
FROM products
WHERE price > (SELECT AVG(price) FROM products);
': near "FROM": syntax error

---
## 🧠 Zadania wyzwanie (13–20)

### 🧠 Zadanie 13 – Analiza sprzedaży miesięcznej

Utwórz tabelę `sales` z kolumnami: `sale_id`, `product`, `category`, `sale_date`, `revenue`.

**Wymagania:**
- Dodaj 1000 losowych transakcji z dat 2024-01-01 do 2024-12-31
- Pogrupuj według miesiąca (użyj `strftime('%Y-%m', sale_date)`)
- Oblicz miesięczny przychód, liczbę transakcji, średnią wartość transakcji
- Stwórz wykres liniowy pokazujący trend przychodów w czasie
- Znajdź miesiące z przychodem powyżej średniej

**Oczekiwany rezultat:**
- Tabela z podsumowaniem miesięcznym
- Wykres trendu (line plot)
- Lista "najlepszych" miesięcy

*(challenge)*

### 🧠 Zadanie 14 – Analiza kohortowa klientów

**Wymagania:**
- Utwórz tabelę klientów z `registration_date` i tabelę zamówień
- Pogrupuj klientów według miesiąca rejestracji (kohorta)
- Dla każdej kohorty oblicz:
  - Liczbę klientów
  - Liczbę zamówień
  - Łączny przychód
  - Średnią wartość zamówienia na klienta (LTV — Lifetime Value)
- Wizualizacja: heatmapa kohort

**Dataset:** Własny

*(challenge)*

### 🧠 Zadanie 15 – Window Functions (zaawansowane)

**Wymagania:**
- Użyj funkcji okienkowych (RANK, ROW_NUMBER) do rankingu produktów
- Dla każdej kategorii znajdź TOP 3 najlepiej sprzedające się produkty
- Zapytanie: `RANK() OVER (PARTITION BY category ORDER BY revenue DESC)`
- Wyświetl: kategoria, produkt, przychód, ranking w kategorii

**Dataset:** Sales/Products

*(challenge)*

### 🧠 Zadanie 16 – Optymalizacja zapytań

**Wymagania:**
- Utwórz tabelę z 100 000 wierszami
- Napisz złożone zapytanie (JOIN + WHERE + GROUP BY + HAVING + ORDER BY)
- Zmierz czas wykonania **BEZ** indeksów
- Zidentyfikuj kolumny do indeksowania (użyte w WHERE, JOIN, GROUP BY)
- Utwórz indeksy i zmierz ponownie
- Raport: które indeksy dały największe przyspieszenie

**Oczekiwany rezultat:**
- Porównanie czasów wykonania
- Lista utworzonych indeksów i ich wpływ

**Dataset:** Własny (duży)

*(challenge)*

> Osobne połączenie `conn_big`.

### 🧠 Zadanie 17 – Analiza RFM (Recency, Frequency, Monetary)

**Wymagania:**
- **Dataset:** Customers + Orders
- Dla każdego klienta oblicz:
  - **Recency:** Ile dni od ostatniego zamówienia
  - **Frequency:** Liczba zamówień
  - **Monetary:** Łączna wartość zamówień
- Użyj CASE WHEN do klasyfikacji klientów (VIP, Regular, At-risk, Lost)
- Wizualizacja: scatter plot Frequency vs Monetary z kolorami według segmentu

**Dataset:** E-commerce

*(challenge)*

### 🧠 Zadanie 18 – Self-JOIN

**Wymagania:**
- Utwórz tabelę `employees` z kolumnami: `employee_id`, `name`, `manager_id`
- Użyj self-JOIN aby wyświetlić pracownika i jego managera:
  `SELECT e1.name AS employee, e2.name AS manager FROM employees e1 LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id`
- Znajdź wszystkich pracowników bez managera (TOP management)
- Policz liczbę podwładnych dla każdego managera

**Dataset:** Własny (hierarchia)

*(challenge)*

In [52]:
employee_manager = pd.read_sql_query("""
    SELECT e1.name AS employee, e2.name AS manager
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
""", conn)

employee_no_manager = pd.read_sql_query("""
    SELECT e1.name AS employee
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
    WHERE e2.employee_id IS NULL
""", conn)

manager_employee_count = pd.read_sql_query("""
    SELECT e2.name AS manager, COUNT(e1.employee_id) AS num_employees
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
    GROUP BY e2.employee_id, e2.name
""", conn)

employee_manager, employee_no_manager, manager_employee_count

(         employee        manager
 0   Anna Dyrektor            NaN
 1       Piotr CFO  Anna Dyrektor
 2       Marek CTO  Anna Dyrektor
 3    Ewa Dev Lead      Marek CTO
 4    Jan Analityk      Piotr CFO
 5  Sara Developer   Ewa Dev Lead
 6   Adam Ksiegowy      Piotr CFO
 7         Lena QA   Ewa Dev Lead,
         employee
 0  Anna Dyrektor,
          manager  num_employees
 0            NaN              1
 1  Anna Dyrektor              2
 2      Piotr CFO              2
 3      Marek CTO              1
 4   Ewa Dev Lead              2)

### 🧠 Zadanie 19 – Pivot Table w SQL

**Wymagania:**
- **Dataset:** Sales z kolumnami (date, category, revenue)
- Stwórz "pivot table" pokazującą przychód dla każdej kategorii w każdym miesiącu
- Użyj CASE WHEN i GROUP BY:

```sql
SELECT
    month,
    SUM(CASE WHEN category='A' THEN revenue ELSE 0 END) AS cat_A,
    SUM(CASE WHEN category='B' THEN revenue ELSE 0 END) AS cat_B
FROM sales
GROUP BY month
```
- Wizualizacja: stacked bar chart

**Dataset:** Sales

*(challenge)*

### 🧠 Zadanie 20 – Green SQL Challenge

**Wymagania:**
- Utwórz bazę z 500 000 rekordami (symulacja produkcji)
- Napisz 3 wersje tego samego zapytania analitycznego:
  1. ❌ Nieoptymalne (bez indeksów, SELECT *, filtrowanie w Pythonie)
  2. ⚠️ Częściowo optymalne (indeksy, ale agregacja w Pythonie)
  3. ✅ Pełna optymalizacja (indeksy, WHERE, GROUP BY, LIMIT w SQL)
- Zmierz:
  - Czas wykonania
  - Zużycie pamięci (`memory_usage`)
  - Liczba przesłanych danych (transfer)
- Raport: Oszczędność energii (czas × zużycie pamięci jako proxy)

**Oczekiwany rezultat:**
- 3 skrypty z benchmarkami
- Wykres porównawczy (bar chart: czas, pamięć, transfer)
- Wnioski o best practices Green SQL

**Dataset:** Własny (duży)

*(challenge)*

> Osobne połączenie `conn_green`.